In [4]:
import csv
import json
import logging

logging.basicConfig(level=logging.INFO)

def make_report(csv_path: str, json_path : str) -> int :
    try :
        score = [] # json 파일로 넣기 전 정리할 딕셔너리

        with open("scores.csv", "r", encoding = "utf-8") as f :
            reader = csv.DictReader(f) #DictReader는 딕셔너리로 다룬다는 것
            
            for row in reader : # 한 객체씩 읽어옴
                if "" in [row["중간"], row["기말"], row["과제"]] : # 셋 중 하나라도 0이라면
                    average = None
                    grade = None
                else :
                    average = round((int(row["중간"])*3 + int(row["기말"])*5 + int(row["과제"])*2) / 10, 1) #gpt

                    if average >= 90 : # None값이 아닐 때만 작동하게
                        grade = "A"
                    elif (80 <= average) :
                        grade = "B"
                    elif 70 <= average :
                        grade = "C"
                    else :
                        grade = "F"

                result = {
                    "이름" : row["이름"],
                    "학번" : row["학번"],
                    "점수" : {f"중간": int(row["중간"]) if row["중간"] else None, "기말": int(row["기말"]) if row["기말"] else None, "과제": int(row["과제"]) if row["과제"] else None},
                    "평균" : average,
                    "등급" : grade
                }

                score.append(result)
                logging.info(f"{row['이름']}: 평균 {average}, 등급 {grade}") # info랑 INFO 차이

            with open("student.json", "w", encoding = "utf-8") as f:
                json.dump(score, f,
                ensure_ascii = False,
                indent=2)
        
        return len(score)


    except FileNotFoundError:
        logging.warning("파일이 존재하지 않음")
        return 0

    except UnicodeDecodeError:
        logging.error("인코딩이 잘못됨")
        return 0

make_report("scores.csv", "student.json")

INFO:root:김언어: 평균 89.5, 등급 B
INFO:root:이국문: 평균 84.4, 등급 B
INFO:root:박영문: 평균 93.5, 등급 A
INFO:root:최역사: 평균 None, 등급 None


4

코드 실행 결과

INFO:root:김언어: 평균 89.5, 등급 B
INFO:root:이국문: 평균 84.4, 등급 B
INFO:root:박영문: 평균 93.5, 등급 A
INFO:root:최역사: 평균 None, 등급 None

4

로그 레벨을 INFO로 설정하여 디버그 단계는 보이지 않게 한다.
정리된 정보를 입력할 score 딕셔너리를 만들고, with문으로 파일을 자동으로 닫을 수 있게 해준다.
for문으로 하나씩 읽어와서 비어있다면 None값을, 아니라면 평균과 등급을 계산한다.
이후 딕셔너리로 정리하여 score에 저장한다.
dump로 json파일로 변환하고 score의 길이를 변환한다(학생수)

위 과정을 try문으로 감싸고 except문에 filenotfounderror와 unicodedecodeerror를 둬서 각각의 오류일 경우 잘못된 이유와 함께 0을 반환한다.


In [ ]:
class InvalidJamoError(ValueError):
    """한글이 아닌 입력을 제한"""

def classify_jamo(c : str) -> str :
    if not isinstance(c, str) :
        raise TypeError(f"'{c}'는 문자열이 아닙니다.")
    if len(c) != 1 :
        raise ValueError(f"'{c}'는 길이가 1이 아닙니다")
    if 0x3131 <= ord(c) <= 0x314E :
        return "자음"
    if 0x314F <= ord(c) <= 0x3163 :
        return "모음"
    raise InvalidJamoError(f"'{c}'는 한글이 아님")


inputs = ["ㄱ", "ㅏ", "ㄲ", "가", "AB", 5, "ㅎ", "|", ""]

for x in inputs:
    try:
        result = classify_jamo(x)
        print(f"{x} : {result}")

    except TypeError as e:
        print(f"[TypeError] {e}")

    except InvalidJamoError as e:
        print(f"[InvalidJamoError] {e}")

    except ValueError as e:
        print(f"[ValueError] {e}")    


ㄱ : 자음
ㅏ : 모음
ㄲ : 자음
[InvalidJamoError] '가'는 한글이 아님
[ValueError] 'AB'는 길이가 1이 아닙니다
[TypeError] '5'는 문자열이 아닙니다.
ㅎ : 자음
[InvalidJamoError] '|'는 한글이 아님
[ValueError] ''는 길이가 1이 아닙니다


실행 결과
ㄱ : 자음
ㅏ : 모음
ㄲ : 자음
[InvalidJamoError] '가'는 한글이 아님
[ValueError] 'AB'는 길이가 1이 아닙니다
[TypeError] '5'는 문자열이 아닙니다.
ㅎ : 자음
[InvalidJamoError] '|'는 한글이 아님
[ValueError] ''는 길이가 1이 아닙니다

InvalidJamoError를 ValueError의 자식 에러로 만든다.
문제 조건대로 if문을 이용해 다섯 가지 경우에 대해 반환 또는 Error를 발생시킨다.
이를 except문에서 각각 받아온다. 이때 InvalidJamoError는 ValueError의 자식이기 때문에 ValueError 위에 써야 한다. 아래 쓸 경우 모두 ValueError로 판단되기 때문이다.
또한 InvalidJamoError는 설명문에 써두었듯이 한글이 아닌 입력일 때 발생하는 에러이기 때문에 프로그램의 오류가 아니라 입력이 올바르지 않을 때 발생하는 에러이다. 따라서 ValueError의 자식으로 두는 것이 적절하다.

gpt사용 : https://chatgpt.com/share/69fd35fb-77f0-8320-9a2a-423a95dbb421